In [1]:
from functools import partial

import jax
import mediapy
from utils import expert_step, run_and_log_scenario
from waymax import dynamics

from vmax import PATH_TO_PROJECT
from vmax.simulator import datasets, make_data_generator, make_env, visualization


# %load_ext autoreload
# %autoreload 2

In [3]:
MAX_NUM_OBJECTS = 64
INCLUDE_SDC_PATHS = True
SEED = 0
path_dataset = "/media/rtgtx7/加速盘/data/validation.tfrecord" # TO FILL


data_generator = make_data_generator(
    path=path_dataset,
    max_num_objects=MAX_NUM_OBJECTS,
    include_sdc_paths=INCLUDE_SDC_PATHS,
    seed=SEED,
)


env = make_env(
    max_num_objects=MAX_NUM_OBJECTS,
    dynamics_model=dynamics.InvertibleBicycleModel(normalize_actions=True),
    observation_type="gt",
    observation_config={
        "path_target": {
            "features": ["waypoints"],
            "num_points": 20,
            "points_gap": 1,
        },
    },
)

ValueError: Metric run_red_light has already been registered.

In [3]:
scenario = next(data_generator)
simulator_state = env.reset(scenario)

W0000 00:00:1765608058.621172  552102 gpu_device.cc:2431] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1765608058.626762  552102 gpu_device.cc:2431] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1765608058.709705  552102 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13214 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5070 Ti, pci bus id: 0000:01:00.0, compute capability: 12.0


NotFoundError: {{function_node __wrapped__IteratorGetNext_output_types_36_device_/job:localhost/replica:0/task:0/device:CPU:0}} /home/rtgtx7/Desktop/AD/v-max/data/scenariomax/validation_tfexample/training.tfrecord; No such file or directory [Op:IteratorGetNext] name: 

In [ ]:
import os, sys
os.environ["JAX_PLATFORMS"] = "cpu"  # do this BEFORE importing jax anywhere
# (then import v-max / waymax code that pulls in jax)
from vmax.simulator.visualization import viz as visualization
img = visualization.plot_input_agent(simulator_state, env)


In [1]:
img = visualization.plot_input_agent(simulator_state, env)
mediapy.show_image(img)

NameError: name 'visualization' is not defined

In [ ]:
_step = partial(expert_step, env)
_jitted_expert_step = jax.jit(_step)

In [ ]:
# for _i in range(5):
scenario = next(data_generator)
imgs = run_and_log_scenario(env, scenario, _jitted_expert_step)
mediapy.show_video(imgs, fps=10)